# Extracción sistemática

Una de las principales albores de los ingenieros de datos es la obtención sistemática de datos para poblar nuestros sistemas de analítica. Para ello existen herramientas especializadas que no solo pueden leer de sistemas corporativos (Bases de Datos) si no también de APIs. Aquí veremos una solución de nueva generación que nos ofrece multitud de opciones.

In [2]:
#!pip install dlt

**Data Load Tool** se emplea para la obtención e inserción de los datos en sistemas que controlemos nosotros. A modo sencillo podemos ver cómo podemos por ejemplo obtener datos de una API (hemos limitado el número de datos para que no se haga largo).

En este caso consultaremos una versión paginada de la API de pokemon... https://pokeapi.co/docs/v2#info

In [3]:
from dlt.sources.helpers.rest_client import RESTClient

# Initialize the RESTClient with the Pokémon API base URL
client = RESTClient(base_url="https://pokeapi.co/api/v2")

# Using the paginate method to automatically handle pagination
num = 10
poke_list = []
for page in client.paginate("/pokemon"):
    poke_list += page
    num -= 1

    # Paramos al obtener 10
    if num == 0:
        break

Con esto podemos montar nuestro dataframe, operar sobre él y cargarlo en un sistema destino.

In [4]:
import pandas as pd

poke_df = pd.DataFrame(poke_list)
poke_df

,name,url
0,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/
1,ivysaur,https://pokeapi.co/api/v2/pokemon/2/
2,venusaur,https://pokeapi.co/api/v2/pokemon/3/
3,charmander,https://pokeapi.co/api/v2/pokemon/4/
4,charmeleon,https://pokeapi.co/api/v2/pokemon/5/
...,...,...
195,espeon,https://pokeapi.co/api/v2/pokemon/196/
196,umbreon,https://pokeapi.co/api/v2/pokemon/197/
197,murkrow,https://pokeapi.co/api/v2/pokemon/198/
198,slowking,https://pokeapi.co/api/v2/pokemon/199/


Los pipelines son la forma de definir esas tuberías de entrada y almacenamiento de datos de manera que se ejecuten de forma continua. Todavía no hemos hablado de las bases de datos pero son un destino frecuente a la hora de definir nuestros destinos.

In [6]:
#!pip install duckdb

In [ ]:
import dlt

pipeline = dlt.pipeline(destination="duckdb", dataset_name="country_data")

data = [
    {'country': 'USA', 'population': 331449281, 'capital': 'Washington, D.C.'},
    {'country': 'Canada', 'population': 38005238, 'capital': 'Ottawa'},
    {'country': 'Germany', 'population': 83019200, 'capital': 'Berlin'}
]

info = pipeline.run(data, table_name="countries")
# se genera un archi duck db (una base de datos)
print(info)

Pipeline dlt_ipykernel_launcher load step completed in 0.34 seconds
1 load package(s) were loaded to destination duckdb and into dataset country_data
The duckdb destination used duckdb:///c:\Users\unaif\OneDrive\Documentos\Documentos_Clase_Data\DSPT2025\2-Data Analysis\5-Fuentes de datos\Web\Teoría\dlt_ipykernel_launcher.duckdb location to store data
Load package 1747420165.950234 is LOADED and contains no failed jobs


In [8]:
info.dataset_name

'country_data'

In [9]:
import duckdb

db = duckdb.connect(database="dlt_ipykernel_launcher.duckdb")
db.sql("DESCRIBE;")

┌────────────────────────┬──────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│        database        │    schema    │        name         │                                           column_names                                           │                                      column_types                                       │ temporary │
│        varchar         │   varchar    │       varchar       │                                            varchar[]                                             │                                        varchar[]                                        │  boolean  │
├────────────────────────┼──────────────┼─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────────────────────────────────

In [10]:
db.sql("SELECT * FROM country_data.countries;")

┌─────────┬────────────┬──────────────────┬───────────────────┬────────────────┐
│ country │ population │     capital      │   _dlt_load_id    │    _dlt_id     │
│ varchar │   int64    │     varchar      │      varchar      │    varchar     │
├─────────┼────────────┼──────────────────┼───────────────────┼────────────────┤
│ USA     │  331449281 │ Washington, D.C. │ 1747420165.950234 │ oBJ9B7tC3J5yLA │
│ Canada  │   38005238 │ Ottawa           │ 1747420165.950234 │ lERkvKVCltzkAQ │
│ Germany │   83019200 │ Berlin           │ 1747420165.950234 │ 44yHhM98IQGGjA │
└─────────┴────────────┴──────────────────┴───────────────────┴────────────────┘

In [11]:
data_df = db.sql("SELECT * FROM country_data.countries;").df()
data_df

,country,population,capital,_dlt_load_id,_dlt_id
0,USA,331449281,"Washington, D.C.",1747420165.950234,oBJ9B7tC3J5yLA
1,Canada,38005238,Ottawa,1747420165.950234,lERkvKVCltzkAQ
2,Germany,83019200,Berlin,1747420165.950234,44yHhM98IQGGjA


In [9]:
type(data_df)

pandas.core.frame.DataFrame

Podemos montar estructuras complejas que se hagan cargo de ir obteniendo los datos periodicamente y formando nuestra base de datos.

In [12]:
import dlt
from dlt.sources.helpers.rest_client import RESTClient
from dlt.sources.helpers.rest_client.paginators import JSONResponsePaginator

github_client = RESTClient(
    base_url="https://pokeapi.co/api/v2",
    paginator=JSONResponsePaginator(next_url_path="next"),
    data_selector="results"
)#pregunta hasta quq no haya mas 

@dlt.resource
def get_pokemons():
    for page in github_client.paginate(
        "/pokemon",
        params={
            "limit": 100,
        },
    ):
        yield page

pipeline = dlt.pipeline(
    pipeline_name="get_pokemons",
    destination="duckdb",
    dataset_name="pokemons",
    progress='log'
)#esto consige los pokemons
load_info = pipeline.run(get_pokemons)
print(load_info)

C:\Users\unaif\AppData\Local\Temp\ipykernel_5824\2448188104.py:7: DeprecationWarning: JSONResponsePaginator is deprecated and will be removed in version 1.0.0. Use JSONLinkPaginator instead.
  paginator=JSONResponsePaginator(next_url_path="next"),


----------------------------- Extract get_pokemons -----------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 164.13 MB (88.20%) | CPU usage: 0.00%

----------------------------- Extract get_pokemons -----------------------------
Resources: 0/1 (0.0%) | Time: 0.19s | Rate: 0.00/s
get_pokemons: 100  | Time: 0.00s | Rate: 0.00/s
Memory usage: 165.16 MB (88.20%) | CPU usage: 0.00%

----------------------------- Extract get_pokemons -----------------------------
Resources: 0/1 (0.0%) | Time: 1.35s | Rate: 0.00/s
get_pokemons: 700  | Time: 1.16s | Rate: 604.44/s
Memory usage: 166.79 MB (88.20%) | CPU usage: 0.00%

----------------------------- Extract get_pokemons -----------------------------
Resources: 0/1 (0.0%) | Time: 2.45s | Rate: 0.00/s
get_pokemons: 1100  | Time: 2.25s | Rate: 487.95/s
Memory usage: 167.50 MB (88.20%) | CPU usage: 0.00%

----------------------------- Extract get_pokemons -----------------------------
Resources: 0/1 (0.0%) | Time: 

In [13]:
db = duckdb.connect(database="get_pokemons.duckdb")
db.sql("DESCRIBE;")

┌──────────────┬──────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│   database   │  schema  │        name         │                                           column_names                                           │                                      column_types                                       │ temporary │
│   varchar    │ varchar  │       varchar       │                                            varchar[]                                             │                                        varchar[]                                        │  boolean  │
├──────────────┼──────────┼─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┼─────────

In [23]:
response = db.sql("SELECT * FROM pokemons.get_pokemons;")
response.to_df()

,name,url,_dlt_load_id,_dlt_id
0,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1716662591.1987376,krLKpOxc4MGH1Q
1,ivysaur,https://pokeapi.co/api/v2/pokemon/2/,1716662591.1987376,dtpkE/WdKcckMA
2,venusaur,https://pokeapi.co/api/v2/pokemon/3/,1716662591.1987376,67nHvGff8mr5BQ
3,charmander,https://pokeapi.co/api/v2/pokemon/4/,1716662591.1987376,xHwmGKB6D7HdWw
4,charmeleon,https://pokeapi.co/api/v2/pokemon/5/,1716662591.1987376,mwU6Hv0psXTvmQ
...,...,...,...,...
1297,ogerpon-wellspring-mask,https://pokeapi.co/api/v2/pokemon/10273/,1716662591.1987376,4ZLlfAZEdTEFrA
1298,ogerpon-hearthflame-mask,https://pokeapi.co/api/v2/pokemon/10274/,1716662591.1987376,/yQJsqybwEEm7A
1299,ogerpon-cornerstone-mask,https://pokeapi.co/api/v2/pokemon/10275/,1716662591.1987376,c+n8mFPEA7YcnA
1300,terapagos-terastal,https://pokeapi.co/api/v2/pokemon/10276/,1716662591.1987376,b7NifCIEsU1YhQ
